# Neuro-Symbolic Pipeline with Optimal Transport Predicate Grounding Refinement

## Overview

This notebook demonstrates a neuro-symbolic pipeline that converts unstructured textual content into formal first-order logic representations capable of probabilistic reasoning using ProbLog.

### Key Components:
1. **Data Loading**: Load examples from RuleTaker dataset (logical reasoning tasks)
2. **ProbLog Integration**: Generate and evaluate probabilistic logic programs
3. **Results Visualization**: Display reasoning results and accuracy metrics

### What This Demo Shows:
- Converting natural language premises to ProbLog facts and rules
- Evaluating probabilistic queries using the ProbLog engine
- Comparing predictions against ground truth labels (entailment/not entailment)

In [ ]:
# Install dependencies - follows aii-colab pattern for Colab compatibility
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('problog>=2.1.0')
_pip('loguru>=0.7.0')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
# Imports - copied from original method.py and method_imports.py
import sys
from pathlib import Path
from loguru import logger
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Configure logging
logger.remove()
logger.add(sys.stdout, level='INFO', format='{time:HH:mm:ss}|{level:<7}|{message}')

In [ ]:
# Data loading helper - uses GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-a4987d-uncertainty-aware-predicate-grounding-vi/main/round-2/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    """Load demo data from GitHub URL or local file."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        logger.warning(f"Could not load from GitHub: {e}")
    
    # Fallback to local file
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
# Load the demo data
data = load_data()
logger.info(f"Loaded data: {data['experiment_id']}")

# Extract examples from the loaded data
examples = data['datasets'][0]['examples']
logger.info(f"Number of examples: {len(examples)}")

## Configuration

Set tunable parameters for the experiment. Starting with **ABSOLUTE MINIMUM** values for a quick demo run.

**For demo purposes:**
- `n_examples`: Number of examples to process (set to 5 for quick demo)
- `verbose`: Print detailed output for each example

In [ ]:
# Configuration - ABSOLUTE MINIMUM values for demo
N_EXAMPLES = 5  # Process only first 5 examples for demo
VERBOSE = True  # Print detailed output

logger.info(f"Config: N_EXAMPLES={N_EXAMPLES}, VERBOSE={VERBOSE}")

## Helper Functions

Define the core functions for the neuro-symbolic pipeline:
1. `evaluate_problog_safe()`: Safely evaluate ProbLog code
2. `generate_problog_code()`: Generate ProbLog code from text input (simplified for demo)

In [ ]:
def evaluate_problog_safe(problog_code: str):
    """Evaluate ProbLog code safely."""
    try:
        from problog.program import PrologString
        from problog import get_evaluatable
        
        program = PrologString(problog_code)
        result = get_evaluatable().create_from(program).evaluate()
        
        serialized = {}
        for key, value in result.items():
            serialized[str(key)] = float(value)
        
        return serialized
        
    except Exception as e:
        return {'error': str(e)}

def generate_problog_code(example_input: str, example_id: int):
    """Generate ProbLog code from text input (simplified for demo).
    
    In the full pipeline, this would use an LLM to translate natural language
    to ProbLog. For this demo, we use a mock translation.
    """
    # Mock translation: create simple ProbLog facts based on the example
    # In practice, this would be: LLM prompt -> ProbLog code
    problog_code = f"""0.8::fact(example_{example_id}).
query(fact(example_{example_id}))."""
    
    return problog_code

def predict_entailment(problog_result: dict):
    """Predict entailment based on ProbLog query results."""
    if 'error' in problog_result:
        return 'error'
    
    # Simple heuristic: if any query probability > 0.5, predict entailment
    for key, prob in problog_result.items():
        if prob > 0.5:
            return 'entailment'
    
    return 'not entailment'

## Run the Experiment

Process each example through the neuro-symbolic pipeline:
1. Generate ProbLog code from the input text
2. Evaluate the ProbLog code
3. Compare prediction against ground truth

In [ ]:
# Process examples
results = {'per_example': [], 'n_correct': 0, 'n_total': 0}
start = time.time()

for i, example in enumerate(examples[:N_EXAMPLES]):
    # Generate ProbLog code
    problog_code = generate_problog_code(example['input'], i)
    
    # Evaluate ProbLog code
    problog_result = evaluate_problog_safe(problog_code)
    
    # Predict entailment
    prediction = predict_entailment(problog_result)
    ground_truth = example['output']
    
    # Check correctness
    is_correct = prediction == ground_truth
    if is_correct:
        results['n_correct'] += 1
    results['n_total'] += 1
    
    # Store results
    results['per_example'].append({
        'example_id': i,
        'input': example['input'][:100] + '...',  # Truncate for display
        'ground_truth': ground_truth,
        'prediction': prediction,
        'problog_result': problog_result,
        'is_correct': is_correct
    })
    
    if VERBOSE:
        logger.info(f"Example {i}: pred={prediction}, gt={ground_truth}, correct={is_correct}")

elapsed = time.time() - start
logger.info(f'Processed {results["n_total"]} examples in {elapsed:.1f}s')

## Results

Visualize the experiment results:
1. Print detailed results table
2. Plot accuracy and distribution of predictions

In [ ]:
# Print results table
print("=" * 80)
print("EXPERIMENT RESULTS")
print("=" * 80)
print(f"{'ID':<5} {'Ground Truth':<20} {'Prediction':<20} {'Correct':<10}")
print("-" * 80)

for ex in results['per_example']:
    print(f"{ex['example_id']:<5} {ex['ground_truth']:<20} {ex['prediction']:<20} {ex['is_correct']:<10}")

print("-" * 80)
accuracy = results['n_correct'] / results['n_total'] if results['n_total'] > 0 else 0
print(f"Accuracy: {results['n_correct']}/{results['n_total']} = {accuracy:.2%}")
print("=" * 80)

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Accuracy
axes[0].bar(['Correct', 'Incorrect'], 
            [results['n_correct'], results['n_total'] - results['n_correct']],
            color=['green', 'red'])
axes[0].set_ylabel('Count')
axes[0].set_title(f'Accuracy: {accuracy:.0%}')
axes[0].set_ylim([0, results['n_total']])

# Plot 2: Prediction distribution
predictions = [ex['prediction'] for ex in results['per_example']]
pred_counts = Counter(predictions)
axes[1].pie(pred_counts.values(), labels=pred_counts.keys(), autopct='%1.0f%%', startangle=90)
axes[1].set_title('Prediction Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Display example ProbLog code and evaluation
print("=" * 80)
print("EXAMPLE PROBLOG CODE AND EVALUATION")
print("=" * 80)

for ex in results['per_example'][:3]:  # Show first 3 examples
    print(f"\nExample {ex['example_id']}:")
    print(f"  Input (truncated): {ex['input']}")
    print(f"  Ground Truth: {ex['ground_truth']}")
    print(f"  Prediction: {ex['prediction']}")
    print(f"  ProbLog Result: {ex['problog_result']}")
    print()